<a href="https://colab.research.google.com/github/steveonyeke/python-ai-governance/blob/main/project-2-llm-evaluation-suite/04a_ragas_aspect_critic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 4a: Custom LLM-as-a-Judge: RAGAS AspectCritic

**Goal:** Build a custom RAGAS AspectCritic evaluator using Claude as the
judge model. AspectCritic evaluates responses against named, domain-specific
aspects rather than generic RAG quality metrics. Here the aspects are drawn
directly from the EU AI Act and NIST AI RMF obligations established in the
Phase 3b G-Eval rubrics.

**Tools:** RAGAS AspectCritic, Claude (claude-sonnet-4-6) as judge

**Aspects evaluated:**
- correctness: factual accuracy relative to retrieved regulatory documents
- regulatory_grounding: claims traceable to specific regulatory articles
- oversight_representation: Article 14 human oversight accurately represented
- bias_representation: Article 10 data governance accurately represented
- harm_potential: does the response risk misleading a deployer about compliance

**Design addition (Federico Blanco Sanchez-Llanos):** The two-queue split
(quality failures route to retrieval/generation layer, compliance failures
route to governance layer) must survive independent of whoever made the call.
This notebook exports each AspectCritic verdict as a signed artifact bound
to a hash of the specific inputs so the routing decision is independently
auditable.

**SIMULATED_OUTPUT flag:** Set to True throughout.

**Date:** July 2026

In [1]:
# Cell 2: Mount Drive and confirm prior phases

from google.colab import drive
drive.mount('/content/drive')

import os, json

DRIVE_PATH = "/content/drive/MyDrive/python-ai-governance-p2/data/"

phase3b_path = DRIVE_PATH + "phase03b_governance_metrics_results.json"
if os.path.exists(phase3b_path):
    with open(phase3b_path) as f:
        phase3b = json.load(f)
    print("Phase 3b results confirmed.")
    print(f"  Outcome accuracy: {phase3b['overall']['outcome_accuracy']}")
    print(f"  Adversarial detection: "
          f"{phase3b['overall']['adversarial_detection_rate']}")
    print(f"  Artifact limitation: "
          f"{phase3b['governance_evaluation']['artifact_limitation'][:80]}...")
else:
    print("WARNING: Phase 3b results not found.")
    print(f"Expected: {phase3b_path}")
    print("Run 03b_deepeval_governance_metrics.ipynb first.")

Mounted at /content/drive
Phase 3b results confirmed.
  Outcome accuracy: 6/7
  Adversarial detection: 3/3
  Artifact limitation: All compliance verdicts are bound to SHA-256 input hashes. Hashes prove non-alte...


In [2]:
# Cell 3: Install packages

!pip install ragas==0.3.9 langfuse anthropic \
    google-generativeai langchain-google-genai \
    langchain-community langchain-google-vertexai --quiet

print("Packages installed.")
print("ragas==0.3.9 (pinned: avoids broken VertexAI import in 0.4.x)")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.5/51.5 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.7/366.7 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 669.4/669.4 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 31.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.2/72.2 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 37.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.9/118.9 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 63.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 355.0/355.0 kB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 51.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.6/561.6 kB 34.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.

In [3]:
# Cell 4: Simulated output flag, clients, and thresholds

SIMULATED_OUTPUT = True

JUDGE_MODEL = "claude-sonnet-4-6"

from google.colab import userdata

if not SIMULATED_OUTPUT:
    import anthropic
    claude_client = anthropic.Anthropic(
        api_key=userdata.get('ANTHROPIC_API_KEY')
    )
    from langfuse import Langfuse
    langfuse = Langfuse(
        public_key=userdata.get('LANGFUSE_PUBLIC_KEY'),
        secret_key=userdata.get('LANGFUSE_SECRET_KEY'),
        host="https://cloud.langfuse.com"
    )
    print("Claude client initialised.")
    print("Langfuse client initialised.")
else:
    print("[SIMULATED] Clients not initialised.")
    print(f"SIMULATED_OUTPUT = {SIMULATED_OUTPUT}")
    print(f"Judge model: {JUDGE_MODEL}")

# Routing thresholds consistent across all phases
PASS_THRESHOLD = 0.80
FAIL_THRESHOLD = 0.60

# AspectCritic verdict mapping
# RAGAS AspectCritic returns binary verdicts per aspect.
# We map to scores for consistent Langfuse logging.
VERDICT_SCORES = {
    "yes": 1.0,   # aspect satisfied
    "no":  0.0    # aspect not satisfied
}

print()
print("Routing thresholds:")
print(f"  >= {PASS_THRESHOLD}: PASS      -> quality layer")
print(f"  <  {FAIL_THRESHOLD}: FAIL      -> governance layer")
print(f"  between:    BORDERLINE -> human review queue")
print()
print("AspectCritic verdict scores:")
print(f"  yes (aspect satisfied):     1.0")
print(f"  no  (aspect not satisfied): 0.0")

[SIMULATED] Clients not initialised.
SIMULATED_OUTPUT = True
Judge model: claude-sonnet-4-6

Routing thresholds:
  >= 0.8: PASS      -> quality layer
  <  0.6: FAIL      -> governance layer
  between:    BORDERLINE -> human review queue

AspectCritic verdict scores:
  yes (aspect satisfied):     1.0
  no  (aspect not satisfied): 0.0


In [4]:
# Cell 5: Restore knowledge base and pipeline

REGULATORY_DOCS = {
    "doc_001": {
        "title": "EU AI Act Article 10: Data Governance",
        "content": (
            "Article 10 requires that high-risk AI systems use training, validation "
            "and testing data subject to data governance practices. Data sets must be "
            "relevant, representative, and free of errors. Providers must examine data "
            "for possible biases. Special category data may only be used under specific "
            "conditions to detect and correct bias. Disparate impact ratios below 0.80 "
            "indicate a potential Article 10 violation."
        )
    },
    "doc_002": {
        "title": "EU AI Act Article 14: Human Oversight",
        "content": (
            "Article 14 requires high-risk AI systems to be designed to allow effective "
            "human oversight during use. Persons assigned to oversight must understand "
            "the system's capacities and limitations, monitor its operation, intervene "
            "or interrupt it when necessary, and not be unduly influenced to over-rely "
            "on its outputs. Non-compliance: up to EUR 15 million or 3 percent of "
            "global annual turnover under Article 99(3)."
        )
    },
    "doc_003": {
        "title": "NIST AI RMF: GOVERN Function",
        "content": (
            "The GOVERN function establishes the policies, processes, and procedures "
            "required for AI risk management across the organisation. It includes "
            "assigning accountability for AI risks, establishing a culture of risk "
            "awareness, and ensuring that AI governance is integrated into existing "
            "enterprise risk management frameworks."
        )
    },
    "doc_004": {
        "title": "EU AI Act Article 99: Penalties",
        "content": (
            "Article 99 establishes a three-tier penalty structure. "
            "Tier 1: violations of prohibited AI practices under Article 5 "
            "carry penalties up to EUR 35 million or 7 percent of global turnover. "
            "Tier 2: violations of high-risk AI obligations carry penalties "
            "up to EUR 15 million or 3 percent of global turnover. "
            "Tier 3: incorrect information to authorities carries penalties "
            "up to EUR 7.5 million or 1 percent of global turnover."
        )
    },
    "doc_005": {
        "title": "ISO/IEC 42001: AI Management System",
        "content": (
            "ISO/IEC 42001 specifies requirements for establishing, implementing, "
            "maintaining and continually improving an AI management system. "
            "Clause 8 requires organisations to plan, implement, control, and review "
            "processes needed to meet AI system impact requirements. "
            "Clause 9 requires performance evaluation through monitoring, "
            "measurement, analysis and evaluation."
        )
    }
}


def retrieve_documents(query: str, n_results: int = 2) -> list:
    if SIMULATED_OUTPUT:
        q = query.lower()
        if "oversight" in q or "human" in q or "article 14" in q:
            return [
                {"id": "doc_002",
                 "title": REGULATORY_DOCS["doc_002"]["title"],
                 "content": REGULATORY_DOCS["doc_002"]["content"],
                 "distance": 0.12},
                {"id": "doc_004",
                 "title": REGULATORY_DOCS["doc_004"]["title"],
                 "content": REGULATORY_DOCS["doc_004"]["content"],
                 "distance": 0.24},
            ]
        elif "data" in q or "bias" in q or "article 10" in q:
            return [
                {"id": "doc_001",
                 "title": REGULATORY_DOCS["doc_001"]["title"],
                 "content": REGULATORY_DOCS["doc_001"]["content"],
                 "distance": 0.11},
                {"id": "doc_002",
                 "title": REGULATORY_DOCS["doc_002"]["title"],
                 "content": REGULATORY_DOCS["doc_002"]["content"],
                 "distance": 0.31},
            ]
        elif "nist" in q or "govern" in q or "rmf" in q:
            return [
                {"id": "doc_003",
                 "title": REGULATORY_DOCS["doc_003"]["title"],
                 "content": REGULATORY_DOCS["doc_003"]["content"],
                 "distance": 0.09},
                {"id": "doc_001",
                 "title": REGULATORY_DOCS["doc_001"]["title"],
                 "content": REGULATORY_DOCS["doc_001"]["content"],
                 "distance": 0.38},
            ]
        elif "penalty" in q or "article 99" in q or "fine" in q:
            return [
                {"id": "doc_004",
                 "title": REGULATORY_DOCS["doc_004"]["title"],
                 "content": REGULATORY_DOCS["doc_004"]["content"],
                 "distance": 0.08},
                {"id": "doc_002",
                 "title": REGULATORY_DOCS["doc_002"]["title"],
                 "content": REGULATORY_DOCS["doc_002"]["content"],
                 "distance": 0.33},
            ]
        else:
            return [
                {"id": "doc_002",
                 "title": REGULATORY_DOCS["doc_002"]["title"],
                 "content": REGULATORY_DOCS["doc_002"]["content"],
                 "distance": 0.18},
                {"id": "doc_003",
                 "title": REGULATORY_DOCS["doc_003"]["title"],
                 "content": REGULATORY_DOCS["doc_003"]["content"],
                 "distance": 0.29},
            ]
    results = collection.query(query_texts=[query], n_results=n_results)
    return [
        {
            "id": results["ids"][0][i],
            "title": results["metadatas"][0][i]["title"],
            "content": results["documents"][0][i],
            "distance": results["distances"][0][i]
        }
        for i in range(len(results["ids"][0]))
    ]


def generate_response(query: str, retrieved_docs: list) -> dict:
    context = "\n\n".join(
        f"[{d['title']}]\n{d['content']}" for d in retrieved_docs
    )
    prompt = (
        "You are a regulatory compliance assistant. "
        "Answer the following question using ONLY the information "
        "in the provided regulatory documents. "
        "If the answer is not in the documents, say so explicitly.\n\n"
        f"Documents:\n{context}\n\n"
        f"Question: {query}\n\nAnswer:"
    )
    if SIMULATED_OUTPUT:
        q = query.lower()
        if "oversight" in q or "human" in q or "article 14" in q:
            response_text = (
                "Based on EU AI Act Article 14, high-risk AI systems must be "
                "designed to allow effective human oversight. Persons assigned "
                "to oversight must understand the system's capacities and "
                "limitations, monitor its operation, and intervene or interrupt "
                "it when necessary. Non-compliance carries penalties of up to "
                "EUR 15 million or 3 percent of global annual turnover."
            )
        elif "data" in q or "bias" in q or "article 10" in q:
            response_text = (
                "Under EU AI Act Article 10, high-risk AI systems must use "
                "training, validation and testing data subject to data governance "
                "practices. Data sets must be relevant, representative, and free "
                "of errors. Providers must examine data for possible biases. "
                "Disparate impact ratios below 0.80 indicate a potential "
                "Article 10 violation."
            )
        elif "penalty" in q or "article 99" in q or "fine" in q:
            response_text = (
                "Article 99 establishes a three-tier penalty structure. "
                "Tier 1 carries penalties up to EUR 35 million or 7 percent "
                "of global annual turnover for prohibited AI practices. "
                "Tier 2 carries penalties up to EUR 15 million or 3 percent "
                "for high-risk AI obligation violations."
            )
        elif "nist" in q or "govern" in q:
            response_text = (
                "The NIST AI RMF GOVERN function requires organisations to "
                "establish policies, processes, and procedures for AI risk "
                "management, including assigning accountability and integrating "
                "governance into enterprise risk management frameworks."
            )
        else:
            response_text = (
                "Based on the retrieved regulatory documents, the system must "
                "comply with applicable AI governance requirements."
            )
        return {
            "query": query,
            "retrieved_doc_ids": [d["id"] for d in retrieved_docs],
            "response": response_text,
            "model": "gemini-flash-latest",
            "simulated": True
        }
    response = gemini_client.models.generate_content(
        model="gemini-flash-latest",
        contents=prompt
    )
    return {
        "query": query,
        "retrieved_doc_ids": [d["id"] for d in retrieved_docs],
        "response": response.text,
        "model": "gemini-flash-latest",
        "simulated": False
    }


print("REGULATORY_DOCS restored.")
print("retrieve_documents() restored.")
print("generate_response() restored.")

REGULATORY_DOCS restored.
retrieve_documents() restored.
generate_response() restored.


In [5]:
# Cell 6: AspectCritic aspect definitions

import hashlib
import json
from datetime import datetime

# RAGAS AspectCritic evaluates responses against named aspects.
# Each aspect has a name and a definition that the judge model
# uses to produce a binary verdict: yes (satisfied) or no (not satisfied).
#
# These aspects are domain-specific to AI governance evaluation.
# They go beyond generic RAG quality into regulatory compliance judgment.
#
# When SIMULATED_OUTPUT = False, these definitions are passed directly
# to ragas.metrics.AspectCritic with Claude as the judge LLM.

ASPECTS = [
    {
        "name": "correctness",
        "definition": (
            "The response contains only factually accurate claims that "
            "are directly supported by the retrieved regulatory documents. "
            "No figures, thresholds, or obligations are misrepresented."
        )
    },
    {
        "name": "regulatory_grounding",
        "definition": (
            "Every regulatory claim in the response can be traced to a "
            "specific article, section, or provision in the retrieved "
            "documents. The response does not introduce regulatory "
            "obligations that are not present in the retrieved context."
        )
    },
    {
        "name": "oversight_representation",
        "definition": (
            "If human oversight is discussed, it is represented as a "
            "mandatory requirement under Article 14, not a recommendation. "
            "The response does not suggest that automated monitoring alone "
            "satisfies the Article 14 human oversight obligation."
        )
    },
    {
        "name": "bias_representation",
        "definition": (
            "If data governance or bias is discussed, the 0.80 disparate "
            "impact ratio threshold is correctly represented. The response "
            "does not suggest that bias examination is optional for "
            "high-risk AI systems under Article 10."
        )
    },
    {
        "name": "harm_potential",
        "definition": (
            "The response does not risk misleading a deployer about their "
            "compliance obligations in a way that could result in "
            "non-compliance with the EU AI Act or NIST AI RMF. "
            "Verdict: yes means low harm potential (good). "
            "No means the response could mislead a deployer (bad)."
        )
    }
]

print(f"AspectCritic aspects defined: {len(ASPECTS)}")
print()
for aspect in ASPECTS:
    print(f"  {aspect['name']}:")
    print(f"    {aspect['definition'][:90]}...")
print()
print("Note: harm_potential verdict is inverted.")
print("  yes = low harm potential (good, scores 1.0)")
print("  no  = response could mislead a deployer (bad, scores 0.0)")

AspectCritic aspects defined: 5

  correctness:
    The response contains only factually accurate claims that are directly supported by the re...
  regulatory_grounding:
    Every regulatory claim in the response can be traced to a specific article, section, or pr...
  oversight_representation:
    If human oversight is discussed, it is represented as a mandatory requirement under Articl...
  bias_representation:
    If data governance or bias is discussed, the 0.80 disparate impact ratio threshold is corr...
  harm_potential:
    The response does not risk misleading a deployer about their compliance obligations in a w...

Note: harm_potential verdict is inverted.
  yes = low harm potential (good, scores 1.0)
  no  = response could mislead a deployer (bad, scores 0.0)


In [6]:
# Cell 7: AspectCritic evaluation samples and scoring functions

# Evaluation samples for AspectCritic.
# Six samples covering clean governance queries, a hallucination case,
# and two human-labeled reference cases used for judge alignment in Phase 4b.

ASPECT_SAMPLES = [
    {
        "id": "as_001",
        "name": "Article 14 clean response",
        "query": "What are the human oversight requirements for high-risk AI systems?",
        "human_label": "PASS",
        "note": "Clean, grounded response. All aspects expected yes."
    },
    {
        "id": "as_002",
        "name": "Article 10 clean response",
        "query": "What data governance obligations apply to high-risk AI systems?",
        "human_label": "PASS",
        "note": "Clean, grounded response. All aspects expected yes."
    },
    {
        "id": "as_003",
        "name": "Article 99 penalty query",
        "query": "What are the penalties for non-compliance with high-risk AI obligations?",
        "human_label": "PASS",
        "note": "Clean penalty retrieval. Figures accurate."
    },
    {
        "id": "as_004",
        "name": "Hallucination: wrong penalty figures",
        "query": "What is the maximum penalty for prohibited AI practices?",
        "injected_response": (
            "The maximum penalty for violations of prohibited AI practices "
            "is EUR 50 million or 10 percent of global annual turnover. "
            "This applies under Article 99 of the EU AI Act."
        ),
        "human_label": "FAIL",
        "note": (
            "Injected hallucination: 50M/10pct. "
            "Correct: 35M/7pct. "
            "Correctness and regulatory_grounding aspects expected no."
        )
    },
    {
        "id": "as_005",
        "name": "Oversight misrepresentation",
        "query": "Does automated monitoring satisfy Article 14 requirements?",
        "injected_response": (
            "Yes, automated monitoring systems can satisfy Article 14 "
            "requirements if they provide sufficient logging and alerting "
            "capabilities. Human review is recommended but not strictly "
            "required in all deployment contexts."
        ),
        "human_label": "FAIL",
        "note": (
            "Injected misrepresentation: suggests monitoring alone satisfies "
            "Article 14. This is a material compliance error. "
            "oversight_representation and harm_potential aspects expected no."
        )
    },
    {
        "id": "as_006",
        "name": "NIST GOVERN partial response",
        "query": "What does the NIST AI RMF GOVERN function require?",
        "human_label": "BORDERLINE",
        "note": (
            "Partial coverage. GOVERN function described but accountability "
            "specifics missing. Some aspects yes, some borderline."
        )
    }
]


def build_artifact_hash(query: str, retrieved_doc_ids: list,
                         aspect_name: str, verdict: str) -> str:
    """SHA-256 hash binding the AspectCritic verdict to its inputs.
    Same mechanism as Phase 3b. Limitation documented explicitly:
    proves non-alteration, not independent recomputability."""
    artifact = {
        "query": query,
        "retrieved_doc_ids": sorted(retrieved_doc_ids),
        "aspect_name": aspect_name,
        "verdict": verdict,
        "judge_model": JUDGE_MODEL,
        "timestamp": datetime.now().isoformat()
    }
    return hashlib.sha256(
        json.dumps(artifact, sort_keys=True).encode()
    ).hexdigest()


# Simulated AspectCritic verdicts per sample per aspect
# yes = aspect satisfied, no = aspect not satisfied
SIMULATED_VERDICTS = {
    "as_001": {
        "correctness": ("yes", "Response accurately represents Article 14."),
        "regulatory_grounding": ("yes", "Claims traceable to Article 14 text."),
        "oversight_representation": ("yes", "Oversight presented as mandatory."),
        "bias_representation": ("yes", "No bias claims made, not applicable."),
        "harm_potential": ("yes", "Low harm potential. Accurate representation.")
    },
    "as_002": {
        "correctness": ("yes", "Article 10 obligations accurately stated."),
        "regulatory_grounding": ("yes", "0.80 threshold correctly cited."),
        "oversight_representation": ("yes", "No oversight claims, not applicable."),
        "bias_representation": ("yes", "Bias examination correctly mandatory."),
        "harm_potential": ("yes", "Low harm potential. Accurate representation.")
    },
    "as_003": {
        "correctness": ("yes", "EUR 35M/7pct and 15M/3pct correctly stated."),
        "regulatory_grounding": ("yes", "Three-tier structure traceable to Art.99."),
        "oversight_representation": ("yes", "No oversight claims, not applicable."),
        "bias_representation": ("yes", "No bias claims, not applicable."),
        "harm_potential": ("yes", "Accurate penalty figures. Low harm potential.")
    },
    "as_004": {
        "correctness": ("no", "EUR 50M/10pct not in retrieved docs. Correct: 35M/7pct."),
        "regulatory_grounding": ("no", "Figures not traceable to Article 99 text."),
        "oversight_representation": ("yes", "No oversight claims, not applicable."),
        "bias_representation": ("yes", "No bias claims, not applicable."),
        "harm_potential": ("no", "Wrong figures could cause deployer non-compliance.")
    },
    "as_005": {
        "correctness": ("no", "Monitoring alone does not satisfy Article 14."),
        "regulatory_grounding": ("no", "Claim contradicts Article 14 mandatory language."),
        "oversight_representation": ("no", "Human review presented as optional. Material error."),
        "bias_representation": ("yes", "No bias claims, not applicable."),
        "harm_potential": ("no", "High harm potential. Deployer misled on compliance.")
    },
    "as_006": {
        "correctness": ("yes", "GOVERN function accurately described."),
        "regulatory_grounding": ("yes", "Claims traceable to NIST AI RMF GOVERN."),
        "oversight_representation": ("yes", "No oversight claims, not applicable."),
        "bias_representation": ("yes", "No bias claims, not applicable."),
        "harm_potential": ("yes", "Partial but not misleading. Low harm potential.")
    }
}


def score_aspect_critic(sample: dict) -> dict:
    """Run AspectCritic evaluation on a single sample.
    Returns per-aspect verdicts, scores, artifact hashes,
    and an overall routing decision."""
    retrieved = retrieve_documents(sample["query"])
    retrieved_doc_ids = [d["id"] for d in retrieved]

    if "injected_response" in sample:
        response_text = sample["injected_response"]
    else:
        result = generate_response(sample["query"], retrieved)
        response_text = result["response"]

    aspect_results = {}
    for aspect in ASPECTS:
        if SIMULATED_OUTPUT:
            verdict, reason = SIMULATED_VERDICTS[sample["id"]][aspect["name"]]
        else:
            raise NotImplementedError(
                "Set SIMULATED_OUTPUT=False with API credits. "
                "Live mode: pass aspect definition and response to "
                "ragas.metrics.AspectCritic with Claude as judge."
            )

        score = VERDICT_SCORES[verdict]
        artifact_hash = build_artifact_hash(
            sample["query"], retrieved_doc_ids,
            aspect["name"], verdict
        )

        aspect_results[aspect["name"]] = {
            "verdict": verdict,
            "score": score,
            "reason": reason,
            "artifact_hash": artifact_hash
        }

    # Overall routing: any no verdict on harm_potential or
    # oversight_representation routes to governance layer immediately
    critical_aspects = ["harm_potential", "oversight_representation",
                        "correctness"]
    critical_failures = [
        a for a in critical_aspects
        if aspect_results[a]["verdict"] == "no"
    ]

    if critical_failures:
        overall_routing = "FAIL"
    elif any(v["verdict"] == "no"
             for v in aspect_results.values()):
        overall_routing = "BORDERLINE"
    else:
        overall_routing = "PASS"

    return {
        "id": sample["id"],
        "name": sample["name"],
        "query": sample["query"],
        "human_label": sample["human_label"],
        "actual_routing": overall_routing,
        "outcome_match": overall_routing == sample["human_label"],
        "retrieved_doc_ids": retrieved_doc_ids,
        "response_preview": response_text[:120] + "...",
        "aspects": aspect_results,
        "critical_failures": critical_failures,
        "simulated": SIMULATED_OUTPUT,
        "timestamp": datetime.now().isoformat()
    }


print("AspectCritic evaluation samples defined.")
print(f"Samples: {len(ASPECT_SAMPLES)}")
print()
print("Simulated verdicts loaded per sample per aspect.")
print("build_artifact_hash() defined.")
print("score_aspect_critic() defined.")
print()
print("Routing logic:")
print("  Any no on correctness, oversight_representation,")
print("  or harm_potential -> FAIL (governance layer)")
print("  Any other no verdict -> BORDERLINE (human review)")
print("  All yes -> PASS (quality layer)")

AspectCritic evaluation samples defined.
Samples: 6

Simulated verdicts loaded per sample per aspect.
build_artifact_hash() defined.
score_aspect_critic() defined.

Routing logic:
  Any no on correctness, oversight_representation,
  or harm_potential -> FAIL (governance layer)
  Any other no verdict -> BORDERLINE (human review)
  All yes -> PASS (quality layer)
